In [15]:
import torch
from torch import nn
from torchvision import datasets
from torchvision import transforms
import torch.nn.functional as F


In [ ]:
from torch.utils.data import DataLoader,Subset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
training_data=datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)
test_data=datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)
train_subset = Subset(training_data, range(5000))
train_dataloader=DataLoader(training_data,batch_size=16)
test_dataloader=DataLoader(test_data,batch_size=16)

100%|██████████| 9.91M/9.91M [00:00<00:00, 58.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.64MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.04MB/s]


In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, inp, outp):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(inp, outp) * 0.01)
        self.biases = nn.Parameter(torch.zeros(outp))

    def forward(self, inp):
        return inp @ self.weight + self.biases



In [ ]:
class ConvLayer(nn.Module):
    def __init__(self, inp, outp, kernel_size, stride=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

        self.weight = nn.Parameter(
            torch.randn(outp, inp, kernel_size, kernel_size) * 0.01
        )
        self.biases = nn.Parameter(torch.zeros(outp))

    def forward(self, x):
        B, C, H, W = x.shape
        k = self.kernel_size
        s = self.stride

        H_out = (H - k) // s + 1
        W_out = (W - k) // s + 1

        x_unfold = torch.nn.functional.unfold(x, kernel_size=k, stride=s )

        weight_flat = self.weight.view(self.weight.size(0), -1)

        out = torch.matmul(x_unfold.transpose(1, 2),weight_flat.t())
        out = out + self.biases

        out = out.transpose(1, 2).view(B, self.weight.size(0), H_out, W_out)
        return out




In [ ]:
class MaxPoolLayer(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.k = kernel_size
        self.s = stride

    def forward(self, x):
        B, C, H, W = x.shape
        k = self.k
        s = self.s

        H_out = (H - k) // s + 1
        W_out = (W - k) // s + 1
        x_reshaped = x.reshape(B * C, 1, H, W)
        x_unfold = torch.nn.functional.unfold(x_reshaped, kernel_size=k, stride=s)

        out = x_unfold.max(dim=1)[0]

        out = out.view(B, C, H_out, W_out)

        return out





In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1=ConvLayer(1,4,kernel_size=3)
    self.pool=MaxPoolLayer(kernel_size=2,stride=2)

    self.fc1=DenseLayer(4*13*13,64)
    self.fc2=DenseLayer(64,10)

  def forward(self,x):
    x=self.conv1(x)
    x=F.relu(x)
    x=self.pool(x)

    x = x.view(x.size(0), -1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=CNN().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)



In [ ]:
epochs=2

for epoch in range(epochs):
  model.train()
  correct=0
  total=0
  running_loss=0

  for images,labels in train_dataloader:
    images = images.to(device)
    labels=labels.to(device)

    optimizer.zero_grad()
    outputs=model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    _, preds = torch.max(outputs, 1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

  acc = 100 * correct / total
  print(f"Epoch {epoch+1}, Loss: {running_loss:.3f}, Accuracy: {acc:.2f}%")


Epoch 1, Loss: 1043.033, Accuracy: 91.56%
Epoch 2, Loss: 397.960, Accuracy: 96.81%


In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 95.23%


In [16]:
class Relu_softplus(nn.Module):
  def __init__(self,alpha=0.1):
    super().__init__()
    self.alpha=alpha

  def forward(self,x):
    positive_part=x
    negative_part=self.alpha*torch.log1p(torch.exp(x))
    output=torch.where(x>=0,positive_part,negative_part)
    return output



In [17]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
activation=Relu_softplus(alpha=0.1)
y=activation(x)
print(y)

tensor([0.0127, 0.0313, 0.0000, 1.0000, 2.0000])
